# Solving a task with the exact reference

Every PASSAGE label claims to be an exact optimum. This notebook shows how
to check that claim yourself. It solves the six variants of one scene again with the exact float64 A\*
reference of the code repository, compares the result with the stored labels, shows why the search
budget matters, and then repeats the check on a sample of validation scenes.

**Run it yourself.** From a checkout of the [code repository](https://github.com/ThalesGroup/passage), start Jupyter with `uv run --frozen jupyter lab`, open this notebook and point `dataset_path` to a local copy of [the dataset](https://huggingface.co/datasets/thalesgroup/passage). With `dataset_path = None`, rows are read from the Hub at the pinned `dataset_revision` instead. It runs a few hundred exact searches; about two minutes at 256×256.

In [ ]:
resolution = 256
split = "validation"
sample_id = None  # scene of Sections 1 to 3; None takes the first scene whose routes span a quarter of the side
check_tasks = 16  # validation scenes of Section 4
max_expansions = resolution ** 2  # one expansion per cell lets every exact search finish
output_dir = "notebook-artifacts/notebooks/solve"
dataset_path = "dataset"  # local copy of the dataset; None reads the Hub at dataset_revision
dataset_repo = "thalesgroup/passage"
dataset_revision = "2ab44d899652c7fb99c425488e84a7d565963547"  # verified PASSAGE v1.0 upload
project_path = None  # checkout root; None searches upwards from the working directory

In [ ]:
from pathlib import Path
import json  # noqa: F401
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np  # noqa: F401
import pandas as pd
from IPython.display import Markdown, display

# Locate the code checkout (pyproject.toml next to src/passage) without changing directory.
project_root = Path(project_path).resolve() if project_path else next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src/passage").is_dir()), Path.cwd())
for extra in (project_root / "src", project_root / "scripts"):
    if extra.is_dir() and str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from passage.notebook_reports import make_source, resolve_path, save_figure, write_report  # noqa: E402, F401

source = make_source(dataset_path, dataset_repo=dataset_repo, dataset_revision=dataset_revision,
                     project_root=project_root)
output_path = resolve_path(output_dir, project_root)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)


def relative(path):
    """Show saved files relative to the checkout, never as machine paths."""
    path = Path(path)
    try:
        return path.resolve().relative_to(project_root.resolve()).as_posix()
    except ValueError:
        return path.name


def show(frame, caption=None, floatfmt="{:,.3f}"):
    """Display a table with readable numbers and an optional caption."""
    if caption:
        display(Markdown(f"*{caption}*"))
    floats = list(frame.select_dtypes("float").columns)
    ints = list(frame.select_dtypes("integer").columns)
    others = [c for c in frame.columns if c not in floats and c not in ints]
    styler = frame.style.hide(axis="index")
    for columns, fmt in ((floats, floatfmt), (ints, "{:,}"), (others, None)):
        if columns:
            styler = styler.format(fmt, subset=columns, na_rep="—")
    display(styler)


# One chart style for every PASSAGE notebook: recessive axes, light grid, left-aligned titles.
INK, MUTED, GRID = "#243537", "#5c6765", "#e3e6e3"
OBJECTIVE_COLORS = {"elevation": "#1767ac", "slope_uphill": "#d56418", "energy": "#a32b8e"}
OBJECTIVE_LABELS = {"elevation": "Elevation", "slope_uphill": "Slope (uphill)", "energy": "Energy"}
SPLIT_COLORS = {"train": "#1f8a70", "calibration": "#b07d1a", "validation": "#3a62b5", "test": "#c0392b"}
REGIME_COLORS = {"free": "#238b45", "obstacles": "#1767ac"}
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": "#8a9290", "axes.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True, "axes.titlelocation": "left",
    "axes.titlesize": 10.5, "axes.titleweight": "semibold", "axes.titlecolor": INK, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False, "lines.linewidth": 1.6,
    "figure.dpi": 110, "savefig.dpi": 140, "pdf.fonttype": 42, "ps.fonttype": 42,
})


def figure_title(fig, title, subtitle=None):
    """Left-aligned title and subtitle in a band reserved at the top of the figure."""
    height = fig.get_figheight()
    fig.suptitle(title, x=0.01, y=1 - 0.1 / height, ha="left", va="top", fontsize=12, fontweight="semibold", color=INK)
    if subtitle:
        fig.text(0.01, 1 - 0.4 / height, subtitle, ha="left", va="top", fontsize=8.5, color=MUTED)
    fig.subplots_adjust(top=min(fig.subplotpars.top, 1 - 0.95 / height))


# The shared trajectory style of the paper: one terrain scale, white start circle and goal star,
# green solid routes on free terrain and blue short dashes when obstacles are blocked.
from trajectory_style import render_cost_comparison, draw_comparison, comparison_handles  # noqa: E402, F401
from passage.plotting import decode_scene, render_scene, terrain_colormap, OBSTACLE_COLOR  # noqa: E402, F401


def first_scene(resolution, split, min_share=0.25, min_coverage=0.05):
    """Selection rule fixed in advance: the first scene of the split whose six stored routes all span at least
    `min_share` of the grid side and whose obstacles cover at least `min_coverage` of the cells."""
    for candidate in source.iter_rows(resolution, split, columns=["id", "metadata"], samples=False):
        lengths = [variant["length"] for paths in candidate["metadata"]["paths"].values()
                   for variant in (paths["free"], paths["obstacles"])]
        coverage = candidate["metadata"]["obstacles"]["ratio_actual"]
        if min(lengths) >= min_share * resolution and coverage >= min_coverage:
            return candidate["id"]
    raise ValueError("No scene of the split meets the selection rule")

## 1. Six exact searches on one scene

`exact_reference` runs a deterministic A\* search on the frozen graph: eight-connected, 30 m cells,
obstacles blocked. It returns a status, the optimal cost and route, and the number of cells it
expanded. A search can only claim an optimum if it reaches the goal within its budget.

In [ ]:
import time
from passage.audit import exact_reference

SOLVERS = {"astar_elevation": "elevation", "astar_slope": "slope_uphill", "astar_energy": "energy"}
row = source.get_row(resolution, split, sample_id=sample_id or first_scene(resolution, split), samples=False)
elevation, obstacles, start, goal = decode_scene(row)
meta = row["metadata"]
results, solved = [], {}
for solver, objective in SOLVERS.items():
    for condition in ("free", "obstacles"):
        mask = obstacles.astype(np.uint8) if condition == "obstacles" else np.zeros_like(obstacles, dtype=np.uint8)
        began = time.perf_counter()
        reference = exact_reference(elevation, mask, tuple(map(int, start)), tuple(map(int, goal)),
                                    cost_model=objective, weight=float(meta["solvers"][solver]["weight"]),
                                    max_expansions=max_expansions)
        seconds = time.perf_counter() - began
        stored_route = [tuple(p) for p in row[f"path_{solver}_{condition}"]]
        stored_cost = meta["paths"][solver][condition]["cost"]
        solved[(objective, condition)] = reference
        results.append({
            "Objective": OBJECTIVE_LABELS[objective], "Condition": condition, "Search outcome": reference["status"],
            "Expansions": reference["expansions"], "Seconds": seconds, "Stored cost": stored_cost,
            "Re-solved cost": reference["cost"],
            "Same cost": bool(reference["status"] == "exact" and np.isclose(stored_cost, reference["cost"], rtol=1e-7, atol=1e-6)),
            "Same route": [tuple(p) for p in reference["path"]] == stored_route})
results = pd.DataFrame(results)
show(results, f"Table 1. Stored labels of scene {row['id']} against a fresh exact solve.", "{:,.2f}")

Equal costs prove that the stored route is optimal. A different route with the same cost is a tied
optimum, which the tie policy of the release resolves in a fixed order; the evaluator never penalises
a tied optimum.

## 2. Stored and re-solved routes

Figure 1 overlays both routes for the three obstacle-aware variants, in the paper's comparison style:
the stored label in solid green, the fresh solve in dashed blue. Where they coincide, the dashes sit on
the green line.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11.5, 4.3))
fig.subplots_adjust(top=0.8, bottom=0.14, wspace=0.06)
for ax, (solver, objective) in zip(axes, SOLVERS.items()):
    draw_comparison(elevation, obstacles, tuple(start), tuple(goal), row[f"path_{solver}_obstacles"],
                    solved[(objective, "obstacles")]["path"], ax=ax, labels=("Stored label", "Re-solved"))
    ax.set_title(OBJECTIVE_LABELS[objective], color=OBJECTIVE_COLORS[objective], fontsize=10.5)
fig.legend(handles=comparison_handles(("Stored label", "Re-solved")), loc="lower center", ncol=5, fontsize=8.5)
figure_title(fig, "Stored labels and a fresh exact solve", f"Scene {row['id']} · {resolution}×{resolution} · obstacles blocked")
figures = save_figure(fig, output_path, f"solve-{resolution}-{row['id']}")
plt.show()

*Figure 1. Stored label (solid green) and re-solved route (dashed blue) for each objective, obstacles
blocked.*

## 3. Why the search budget matters

A budget that is too small stops the search before the goal. The stopped search proves nothing: it
neither confirms nor contradicts the label. Table 2 repeats one search with growing budgets.

In [ ]:
weight = float(meta["solvers"]["astar_elevation"]["weight"])
budgets = sorted({1_000, 10_000, 100_000, max_expansions})
sweep = []
for budget in budgets:
    reference = exact_reference(elevation, obstacles.astype(np.uint8), tuple(map(int, start)), tuple(map(int, goal)),
                                cost_model="elevation", weight=weight, max_expansions=budget)
    sweep.append({"Budget (expansions)": budget, "Search outcome": reference["status"],
                  "Expansions used": reference["expansions"], "Cost": reference["cost"]})
show(pd.DataFrame(sweep), "Table 2. One search (elevation objective, obstacles) under growing budgets.", "{:,.1f}")

## 4. A sample of validation scenes

The same check runs on the first `check_tasks` validation scenes, six variants each, through the audit
helper of the code repository.

In [ ]:
from passage.notebook_reports import classical_diagnostics_report

check = classical_diagnostics_report(source, resolution, budget=check_tasks, max_expansions=max_expansions)
sample = pd.DataFrame([{
    "Scene": r["sample_id"], "Objective": OBJECTIVE_LABELS.get(r["cost_model"], r["cost_model"]),
    "Condition": r["task_variant"], "Search outcome": r["reference_status"],
    "Same cost": bool(r["reference_status"] == "exact" and np.isclose(r["stored_cost"], r["optimal_cost"], rtol=1e-7, atol=1e-6)),
    "Expansions": int(r["reference_expansions"] or 0)} for r in check["records"]])
summary_table = (sample.groupby(["Objective", "Condition"])
                 .agg(Variants=("Scene", "size"), Exact=("Search outcome", lambda s: int((s == "exact").sum())),
                      **{"Same cost": ("Same cost", "sum"), "Median expansions": ("Expansions", "median")})
                 .reset_index())
show(summary_table, f"Table 3. Exact re-solve of {check_tasks} validation scenes, by variant.", "{:,.0f}")

## Summary

In [ ]:
display(Markdown("\n".join([
    f"- On scene {row['id']}, {int((results['Search outcome'] == 'exact').sum())} of 6 searches proved an optimum; "
    f"{int(results['Same cost'].sum())} match the stored cost and {int(results['Same route'].sum())} follow the same cells.",
    "- " + "; ".join(f"a budget of {s['Budget (expansions)']:,} expansions "
                     + ("lets the elevation search finish" if s["Search outcome"] == "exact" else "stops it before the goal")
                     for s in sweep) + ".",
    f"- Over {check_tasks} validation scenes, {int((sample['Search outcome'] == 'exact').sum())} of {len(sample)} variants "
    f"reached a proved optimum and {int(sample['Same cost'].sum())} match the stored cost.",
])))
report = {"resolution": resolution, "split": split, "scene": row["id"], "one_scene": results.to_dict("records"),
          "budget_sweep": sweep, "sample": summary_table.to_dict("records"), "max_expansions": max_expansions,
          "figures": [relative(f) for f in figures]}

In [ ]:
report_file = write_report(report, output_path, f"solve-{resolution}.json")
display(Markdown(f"The numbers of this run are saved in `{relative(report_file)}`, next to the figures."))

---
*MIT License · Copyright (c) 2026 THALES. The full licence text is in the `LICENSE` file of the repository.*